In [3]:
import pandas as pd

In [5]:
df_eval = pd.read_csv("data/extrait_eval.csv")
df_eval.head()

,satisfaction_employee_environnement,note_evaluation_precedente,niveau_hierarchique_poste,satisfaction_employee_nature_travail,satisfaction_employee_equipe,satisfaction_employee_equilibre_pro_perso,eval_number,note_evaluation_actuelle,heure_supplementaires,augementation_salaire_precedente
0,2,3,2,4,1,1,E_1,3,Oui,11 %
1,3,2,2,2,4,3,E_2,4,Non,23 %
2,4,2,1,3,2,3,E_4,3,Oui,15 %
3,4,3,1,3,3,3,E_5,3,Oui,11 %
4,1,3,1,2,4,3,E_7,3,Non,12 %


In [6]:
df_sirh = pd.read_csv("data/extrait_sirh.csv")
df_sirh.head()

,id_employee,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,annees_dans_l_entreprise,annees_dans_le_poste_actuel
0,1,41,F,5993,Célibataire,Commercial,Cadre Commercial,8,80,8,6,4
1,2,49,M,5130,Marié(e),Consulting,Assistant de Direction,1,80,10,10,7
2,4,37,M,2090,Célibataire,Consulting,Consultant,6,80,7,0,0
3,5,33,F,2909,Marié(e),Consulting,Assistant de Direction,1,80,8,8,7
4,7,27,M,3468,Marié(e),Consulting,Consultant,9,80,6,2,2


In [7]:
df_sondage = pd.read_csv("data/extrait_sondage.csv")
df_sondage.head()

,a_quitte_l_entreprise,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,code_sondage,distance_domicile_travail,niveau_education,domaine_etude,ayant_enfants,frequence_deplacement,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
0,Oui,0,0,1,1,1,2,Infra & Cloud,Y,Occasionnel,0,5
1,Non,1,3,1,2,8,1,Infra & Cloud,Y,Frequent,1,7
2,Oui,0,3,1,4,2,2,Autre,Y,Occasionnel,0,0
3,Non,0,3,1,5,3,4,Infra & Cloud,Y,Frequent,3,0
4,Non,1,3,1,7,2,1,Transformation Digitale,Y,Occasionnel,2,2


In [ ]:
import xgboost as xgb
import optuna
import pandas as pd
import numpy as np

# --- 1. IMPORTS SCIKIT-LEARN ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- 2. VOS DONNÉES BRUTES ET CONFIGURATION ---

# !!! IMPORTANT !!!
# Chargez ici vos données BRUTES (avant tout encodage)
# X_rh = pd.read_csv(...)
# Y_rh = pd.read_csv(...) ou une Series

# --- VOS COLONNES ICI ---
# Remplissez ces listes avec vos VRAIS noms de colonnes
cols_categoriques = ['col_cat_1', 'col_cat_2']  # EXEMPLE À CHANGER
cols_numeriques = ['col_num_1', 'col_num_2']   # EXEMPLE À CHANGER


# --- 3. PARAMÈTRES FIXES (Basés sur Y_rh) ---
# (Ce bloc est correct et complet)
num_classes = Y_rh.nunique()
print(f"Détection de {num_classes} classes uniques.")
if num_classes == 2:
    model_params_fixed = {'objective': 'binary:logistic', 'eval_metric': 'logloss'}
    print("Type de problème : Classification Binaire")
else:
    model_params_fixed = {'objective': 'multi:softmax', 'eval_metric': 'mlogloss', 'num_class': num_classes}
    print(f"Type de problème : Classification Multi-classe ({num_classes} classes)")


# --- 4. LE SPLIT CORRECT (AVANT Prétraitement) ---
# On sépare les données BRUTES
# X_train/y_train pour Optuna et l'entraînement final
# X_test/y_test pour l'évaluation finale (jamais touché avant)
X_train, X_test, y_train, y_test = train_test_split(
    X_rh, Y_rh, 
    test_size=0.25, # 25% pour le test final
    random_state=42, 
    stratify=Y_rh
)
print(f"Total: {len(X_rh)} | Pour Optuna/Train (X_train): {len(X_train)} | Pour Test (X_test): {len(X_test)}")


# --- 5. DÉFINITION DU PRÉPROCESSEUR ---
# (Ce bloc est correct et complet)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_numeriques),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cols_categoriques)
    ],
    remainder='passthrough'
)


# --- 6. FONCTION OBJECTIVE (NON ABRÉGÉE) ---
# C'est la fonction qu'Optuna va appeler

def objective(trial):
    
    # 6a. Split interne à Optuna (sur X_train/y_train)
    X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
        X_train, y_train, 
        test_size=0.25, # Validation sur 25% du bloc X_train
        random_state=42, 
        stratify=y_train
    )
    
    # 6b. Définition des paramètres à tester (COMPLET)
    param_dynamic = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }
    
    # 6c. Combinaison des paramètres
    param = {**model_params_fixed, **param_dynamic, 'random_state': 42}

    # 6d. Création du Pipeline (Préprocesseur + Modèle)
    pipeline_xgb = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model_xgb', xgb.XGBClassifier(**param, early_stopping_rounds=50))
    ])

    # 6e. Entraînement du pipeline
    pipeline_xgb.fit(
        X_train_opt, y_train_opt,
        model_xgb__eval_set=[(X_val_opt, y_val_opt)], # Préfixe 'model_xgb__'
        model_xgb__verbose=False
    )

    # 6f. Prédiction et calcul du score
    preds = pipeline_xgb.predict(X_val_opt)
    f1_weighted = f1_score(y_val_opt, preds, average='weighted')
    
    return f1_weighted

# --- 7. LANCEMENT DE L'ÉTUDE OPTUNA ---
print("\n--- Démarrage de l'optimisation Optuna (Corrigée) ---")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50) # n_trials=50 pour l'exemple

print("\n--- Optimisation Terminée ---")
print(f"Meilleur F1-Score (Validation) : {study.best_value:.4f}")
print("Meilleurs Paramètres Trouvés :", study.best_params)


# --- 8. CRÉATION ET TEST DU 'model' FINAL ---

# 8a. Récupérer les meilleurs paramètres
best_params = study.best_params
best_params.update(model_params_fixed)
best_params['random_state'] = 42

# 8b. Créer votre 'model' final (en tant que Pipeline)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(**best_params)) # 'model' est le nom interne
])

# 8c. Entraîner votre 'model' sur TOUT X_train et y_train
print("\n--- Entraînement du 'model' final (sur X_train) ---")
model.fit(X_train, y_train)
print("Modèle final entraîné.")

# 8d. Évaluation finale sur X_test et y_test (les données jamais vues)
print("\n--- Évaluation finale sur X_test (Le Vrai Score) ---")
y_pred_test = model.predict(X_test)

f1_test_corrected = f1_score(y_test, y_pred_test, average='weighted')
print(f"Score F1 (Test) CORRIGÉ : {f1_test_corrected:.4f}")

# 8e. Vérification de l'overfitting (Train vs Test)
y_pred_train = model.predict(X_train)
f1_train_corrected = f1_score(y_train, y_pred_train, average='weighted')
print(f"Score F1 (Train) CORRIGÉ : {f1_train_corrected:.4f}")

In [ ]:
import xgboost as xgb
import optuna
import pandas as pd
import numpy as np

# --- 1. IMPORTS SCIKIT-LEARN ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- 2. VOS DONNÉES BRUTES ET CONFIGURATION ---

# !!! IMPORTANT : CHARGEZ VOTRE DATAFRAME BRUT ICI !!!
# df = pd.read_csv("votre_fichier.csv") 
# (Je crée un faux 'df' pour l'exemple)
df = pd.DataFrame(np.random.rand(200, 3), columns=['col_num_1', 'col_num_2', 'col_cat_1'])
df['col_cat_1'] = np.random.choice(['A', 'B', 'C'], 200)
df['ma_cible'] = np.random.choice([0, 1], 200, p=[0.7, 0.3])


# --- VOS COLONNES ICI ---
# !!! REMPLISSEZ CES LISTES AVEC VOS VRAIS NOMS DE COLONNES !!!
NOM_COLONNE_CIBLE = 'ma_cible'                  # EXEMPLE À CHANGER
cols_categoriques = ['col_cat_1']               # EXEMPLE À CHANGER
cols_numeriques = ['col_num_1', 'col_num_2']    # EXEMPLE À CHANGER


# --- 3. CRÉATION DE X_rh ET Y_rh (BRUTS) ---
# X_rh contient les features, Y_rh contient la cible
Y_rh = df[NOM_COLONNE_CIBLE]
X_rh = df.drop(columns=[NOM_COLONNE_CIBLE]) # X_rh = toutes les autres colonnes


# --- 4. PARAMÈTRES FIXES (Basés sur Y_rh) ---
num_classes = Y_rh.nunique()
print(f"Détection de {num_classes} classes uniques.")
if num_classes == 2:
    model_params_fixed = {'objective': 'binary:logistic', 'eval_metric': 'logloss'}
    print("Type de problème : Classification Binaire")
else:
    model_params_fixed = {'objective': 'multi:softmax', 'eval_metric': 'mlogloss', 'num_class': num_classes}
    print(f"Type de problème : Classification Multi-classe ({num_classes} classes)")


# --- 5. LE SPLIT CORRECT (AVANT Prétraitement) ---
# On sépare vos données BRUTES (X_rh, Y_rh)
# X_train/y_train pour Optuna et l'entraînement final
# X_test/y_test pour l'évaluation finale
X_train, X_test, y_train, y_test = train_test_split(
    X_rh, Y_rh, 
    test_size=0.25, # 25% pour le test final
    random_state=42, 
    stratify=Y_rh
)
print(f"Total: {len(X_rh)} | Pour Optuna/Train (X_train): {len(X_train)} | Pour Test (X_test): {len(X_test)}")


# --- 6. DÉFINITION DU PRÉPROCESSEUR ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_numeriques),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cols_categoriques)
    ],
    remainder='passthrough'
)


# --- 7. FONCTION OBJECTIVE (Pour Optuna) ---
# Cette fonction travaille sur X_train/y_train
def objective(trial):
    
    # Split interne à Optuna (sur X_train/y_train)
    X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
        X_train, y_train, 
        test_size=0.25, # Validation sur 25% du bloc X_train
        random_state=42, 
        stratify=y_train
    )
    
    # Définition des paramètres à tester
    param_dynamic = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }
    
    param = {**model_params_fixed, **param_dynamic, 'random_state': 42}

    # Création du Pipeline (Préprocesseur + Modèle)
    pipeline_xgb = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model_xgb', xgb.XGBClassifier(**param, early_stopping_rounds=50))
    ])

    # Entraînement du pipeline
    pipeline_xgb.fit(
        X_train_opt, y_train_opt,
        model_xgb__eval_set=[(X_val_opt, y_val_opt)], # Préfixe 'model_xgb__'
        model_xgb__verbose=False
    )

    # Prédiction et calcul du score
    preds = pipeline_xgb.predict(X_val_opt)
    f1_weighted = f1_score(y_val_opt, preds, average='weighted')
    
    return f1_weighted

# --- 8. LANCEMENT DE L'ÉTUDE OPTUNA ---
print("\n--- Démarrage de l'optimisation Optuna (Corrigée) ---")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50) 

print("\n--- Optimisation Terminée ---")
print(f"Meilleur F1-Score (Validation) : {study.best_value:.4f}")
print("Meilleurs Paramètres Trouvés :", study.best_params)


# --- 9. CRÉATION ET TEST DU 'model' FINAL ---

# Récupérer les meilleurs paramètres
best_params = study.best_params
best_params.update(model_params_fixed)
best_params['random_state'] = 42

# Créer votre 'model' final (en tant que Pipeline)
# C'est votre variable 'model'
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(**best_params)) 
])

# Entraîner votre 'model' sur TOUT X_train et y_train
print("\n--- Entraînement du 'model' final (sur X_train) ---")
model.fit(X_train, y_train)
print("Modèle final entraîné.")

# --- 10. ÉVALUATION FINALE (Le Vrai Score) ---
# Évaluation sur X_test et y_test (les données jamais vues)
print("\n--- Évaluation finale sur X_test (Le Vrai Score) ---")
y_pred_test = model.predict(X_test)

f1_test_corrected = f1_score(y_test, y_pred_test, average='weighted')
print(f"Score F1 (Test) CORRIGÉ : {f1_test_corrected:.4f}")

# Vérification de l'overfitting (Train vs Test)
y_pred_train = model.predict(X_train)
f1_train_corrected = f1_score(y_train, y_pred_train, average='weighted')
print(f"Score F1 (Train) CORRIGÉ : {f1_train_corrected:.4f}")